# 🛢️ Análise Exploratória — Preços de Gasolina no Brasil (2022–2026)

---

## Contexto

Este notebook analisa a série histórica de preços de combustíveis no Brasil, com foco na **gasolina comum**.  
Os dados são provenientes da **ANP** (Agência Nacional do Petróleo, Gás Natural e Biocombustíveis), que coleta semanalmente os preços praticados em postos de todo o país.

O dataset cobre o período de **janeiro de 2022 a abril de 2026**, contemplando mais de **960 mil registros** de gasolina comum distribuídos por todos os estados brasileiros.

---

## Objetivos

- 📈 Entender a evolução histórica do preço médio da gasolina no Brasil
- 🗺️ Identificar padrões regionais — quais estados e regiões são sistematicamente mais caros ou mais baratos
- 📊 Analisar a dispersão interna de cada estado — onde o mercado é mais homogêneo ou volátil
- 🔗 Investigar a relação entre o preço do **petróleo Brent** (em R$) e o preço da gasolina no Brasil, com foco em momentos de choque — testando empiricamente o efeito ***rockets and feathers***

---

## Fonte dos Dados

| Dado | Fonte | Frequência |
|------|-------|------------|
| Preço da gasolina por posto | ANP — gov.br/anp | Mensal |
| Petróleo Brent (USD/barril) | Yahoo Finance (yfinance) | Diária |
| Taxa de câmbio USD/BRL | Yahoo Finance (yfinance) | Diária |

## Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
import glob
import requests
from pathlib import Path

In [ ]:
gasolina = pd.read_csv(r"C:\Users\marce\Downloads\gasolina\limpo\gasolina_limpo.csv")
gasolina['DATA'] = pd.to_datetime(gasolina['DATA'])

## 1. Evolução do Preço da Gasolina no Brasil

O primeiro passo é entender o comportamento geral do preço ao longo do tempo.  

In [ ]:
preco_mensal = gasolina.resample('ME', on='DATA')['PRECO'].mean().reset_index()

fig = px.line(preco_mensal, x='DATA', y='PRECO',
              title='Preço médio da gasolina no Brasil',
              labels={'DATA': 'Data', 'PRECO': 'Preço médio (R$/L)'},
              template='plotly_white')
fig.write_html('temporal_brasil.html')
fig.show()

### 📈 Evolução Temporal do Preço

![Preço médio da gasolina no Brasil](MIDIA/TEMPORAL_BRASIL.jpg)

*Figura 1: Comportamento do preço médio da gasolina (R$/L) no território nacional.*

### Observações

O gráfico revela três momentos distintos:

- **Pico de 2022 (até jul/2022):** alta expressiva impulsionada pela guerra na Ucrânia e valorização do dólar, com preço médio atingindo ~R$7,20/L
- **Queda abrupta (jul–dez/2022):** redução do ICMS sobre combustíveis pelo governo federal derrubou o preço em poucos meses, chegando a ~R$4,90/L
- **Retomada gradual (2023–2025):** preços sobem consistentemente, descolando do Brent que permanecia em queda — fenômeno analisado em detalhe na seção de correlação
- **Alta recente (2026):** novo choque geopolítico no Oriente Médio pressiona o Brent e os preços voltam a subir, chegando próximo a R$6,80/L — período analisado com maior granularidade adiante

## 2. Análise por Região


In [ ]:
preco_regiao_mensal = gasolina.groupby(['REGIAO', pd.Grouper(key='DATA', freq='ME')])['PRECO'].mean().reset_index()
media_brasil_mensal = gasolina.resample('ME', on='DATA')['PRECO'].mean().reset_index()
media_brasil_mensal['REGIAO'] = 'BRASIL'
media_brasil_mensal.columns = ['DATA', 'PRECO', 'REGIAO']

df_regiao_plot = pd.concat([preco_regiao_mensal, media_brasil_mensal])

fig.for_each_trace(lambda t: t.update(
    line=dict(
        color="#444444",  # Cinza escuro/grafite
        width=2.5,        # Mais fino que o anterior, mas ainda presente
        dash="solid"
    ) if t.name == "BRASIL" 
    else dict(
        dash="dash", 
        width=1.5         
    )
))

fig.update_layout(
    yaxis_range=[4.5, 8.5], 
    hovermode='x unified',
    template='plotly_white',
    width=1100, height=600
)

fig.show()

### Observações
Teto Regional: A região Norte (N) mantém-se como o teto de preços durante todo o histórico analisado.

Piso Regional: A região Sudeste (SE) apresenta-se consistentemente como a mais barata (piso).

Aderência à Média: O Centro-Oeste (CO) é a região que mais se aproxima da média nacional, com mínima variação em relação ao índice Brasil.

Correlação: Todas as séries temporais apresentam forte correlação, seguindo o mesmo padrão de flutuação e tendência.

## 3. Evolução Espacial dos Preços de Gasolina

In [ ]:
# ── 8. MAPAS ────────────────────────────────────────────────────────
geojson_brasil = requests.get('https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson').json()

# Mapa preço absoluto animado
preco_mensal_estado = gasolina.groupby(['ESTADO', pd.Grouper(key='DATA', freq='ME')])['PRECO'].mean().reset_index()
preco_mensal_estado['DATA_STR'] = preco_mensal_estado['DATA'].dt.strftime('%Y-%m')

fig = px.choropleth(preco_mensal_estado,
                    geojson=geojson_brasil,
                    locations='ESTADO',
                    featureidkey='properties.sigla',
                    color='PRECO',
                    color_continuous_scale='Viridis',
                    animation_frame='DATA_STR',
                    title='Preço médio da gasolina por estado',
                    labels={'PRECO': 'Preço médio (R$/L)', 'ESTADO': 'Estado'},
                    range_color=[4.5, 8.5])
fig.update_geos(fitbounds='locations', visible=False)
fig.update_layout(width=900, height=600)
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 800
fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 300
fig.write_html('mapa_preco.html')
fig.show()



### Observações

O mapa coroplético destaca uma redução  drástica de preços ao longo de todo o território nacional a partir de junho de 2022, reflexo direto da desoneração de ICMS sobre combustíveis (Lei Complementar 194/22)
Além da tendência geral, observam-se alguns outliers significativos no cenário nacional:

Amapá: Destaca-se por apresentar preços consistentemente abaixo da média nacional, um comportamento notável apesar de sua localização na Região Norte.

Acre: Posiciona-se como o "campeão" de preços, figurando consistentemente como o estado com os valores mais altos do Brasil.

Bahia: Também merece destaque no mapa, mantendo preços sistematicamente mais elevados do que os praticados pelos seus vizinhos da Região Nordeste.

## 4. Evolução de diferenças regionais

Este mapa de desvio percentual isola as oscilações nacionais para focar exclusivamente nas desigualdades entre os estados. O objetivo é identificar padrões estruturais e estados que operam fora da média nacional

In [ ]:
# Mapa desvio % animado
media_mensal_brasil = gasolina.groupby(pd.Grouper(key='DATA', freq='ME'))['PRECO'].mean().reset_index()
media_mensal_brasil.columns = ['DATA', 'MEDIA_BRASIL']

desvio_mensal = preco_mensal_estado.merge(media_mensal_brasil, on='DATA')
desvio_mensal['DESVIO_PCT'] = ((desvio_mensal['PRECO'] - desvio_mensal['MEDIA_BRASIL']) / desvio_mensal['MEDIA_BRASIL']) * 100
desvio_mensal['DATA_STR'] = desvio_mensal['DATA'].dt.strftime('%Y-%m')

fig = px.choropleth(desvio_mensal,
                    geojson=geojson_brasil,
                    locations='ESTADO',
                    featureidkey='properties.sigla',
                    color='DESVIO_PCT',
                    color_continuous_scale='RdBu_r',
                    animation_frame='DATA_STR',
                    title='Desvio do preço médio por estado em relação à média nacional (%)',
                    labels={'DESVIO_PCT': 'Desvio (%)', 'ESTADO': 'Estado'},
                    range_color=[-15, 15])
fig.update_geos(fitbounds='locations', visible=False)
fig.update_layout(width=900, height=600)
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 800
fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 300
fig.write_html('mapa_desvio.html')
fig.show()



### Observações


O mapa coroplético de desvio comparado a média nacional permite isolar as flutuações e focar nas disparidades regionais e focar nas disparidades regionais que persistem apesar de quedas ou aumentos glboais nos preços,entre eles destaque-se:
Acre: Mantém-se no limite máximo de desvio (+15%), evidenciando gargalos logísticos que superam o impacto de mudanças tributárias.


Amapá: Destaca-se positivamente na Região Norte, mantendo preços consistentemente abaixo da média nacional.

Bahia: Apresenta desvios sistematicamente superiores aos seus vizinhos de região, indicando uma dinâmica de preços própria.

Mato Grosso: Possui um comportamento estável, mantendo preços sempre muito próximos à média nacional (0% de desvio)

## 5. Distribuição e Outliers (BOXplot)

A análise via Boxplot foca em dois marcos: o pico histórico de junho/2022 e a subida recente atual em março/2026.

In [ ]:


gasolina = gasolina.sort_values(['REGIAO', 'ESTADO'])

fig, axes = plt.subplots(1, 2, figsize=(20, 6))

# Filtro para Junho/2022
pico = gasolina[gasolina['DATA'].dt.to_period('M') == '2022-06']
sns.boxplot(data=pico, x='ESTADO', y='PRECO', hue='REGIAO', ax=axes[0], dodge=False)
axes[0].set_title('Distribuição de preços — Jun/2022 (pico)', fontsize=13)
axes[0].set_xlabel('Estado')
axes[0].set_ylabel('Preço (R$/L)')
axes[0].tick_params(axis='x', rotation=90) # Aumentei a rotação para não sobrepor nomes
axes[0].set_ylim(3, 10)
# Move a legenda para não cobrir os dados
axes[0].legend(title='Região', bbox_to_anchor=(1, 1))

# Filtro para Março/2026
recente = gasolina[gasolina['DATA'].dt.to_period('M') == '2026-03']
sns.boxplot(data=recente, x='ESTADO', y='PRECO', hue='REGIAO', ax=axes[1], dodge=False)
axes[1].set_title('Distribuição de preços — Mar/2026 (recente)', fontsize=13)
axes[1].set_xlabel('Estado')
axes[1].set_ylabel('Preço (R$/L)')
axes[1].tick_params(axis='x', rotation=90)
axes[1].set_ylim(3, 10)
axes[1].legend(title='Região', bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.savefig('boxplot_estado_ordenado.png', dpi=150, bbox_inches='tight')
plt.show()




Extremos Consistentes: O Acre (maiores preços) e o Amapá (menores preços) mantêm suas distribuições totalmente deslocadas do restante do país em ambos os períodos, consolidando-se como os limites estruturais do mercado nacional.

O Caso da Bahia: Apresenta uma das medianas mais altas do Nordeste devido à privatização da Refinaria de Mataripe, que segue política de preços própria. Os outliers inferiores indicam postos que buscam competitividade fora da dominância regional da refinaria.

Estabilidade do MS: O Mato Grosso do Sul mantém a menor dispersão interna e forte aderência à média nacional, sendo o estado mais "previsível" da amostra.

Anomalia de São Paulo: Embora registre uma mediana baixa, o estado apresenta outliers extremos que superam o teto do Acre. Conforme demonstrado na inspeção detalhada na célula abaixo, esses preços se concentram em polos específicos (como o Guarujá e áreas de alto padrão em Barueri), onde a demanda turística e o perfil de consumo local elevam os preços drasticamente acima da média estadual.

In [ ]:
# Top 20 postos mais caros de SP no período recente
gasolina[(gasolina['ESTADO'] == 'SP') & (gasolina['DATA'].dt.to_period('M') == '2026-03')] \
    .sort_values('PRECO', ascending=False) \
    [['MUNICIPIO', 'BANDEIRA', 'PRECO']] \
    .head(20).reset_index(drop=True)

## 6. Dinâmica de Mercado: Risco vs. Preço (Scatter Plot)

Esta análise cruza o Preço Médio (Eixo X) com o Desvio Padrão (Eixo Y) para mapear o perfil de cada estado em Junho/2022 e Março/2026. A visualização divide os mercados em quatro quadrantes operacionais:

Alto Preço / Alta Volatilidade: Estados com combustíveis caros e forte variação interna de preços.

Alto Preço / Baixa Volatilidade: Mercados caros, porém com preços homogêneos entre os postos.

Baixo Preço / Alta Volatilidade: Estados competitivos na média, mas com grandes disparidades regionais.

Baixo Preço / Baixa Volatilidade: Cenário de maior estabilidade e equilíbrio para o consumidor.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Preparação dos dados
scatter_data = gasolina.groupby(
    ['REGIAO', 'ESTADO', pd.Grouper(key='DATA', freq='ME')]
)['PRECO'].agg(['mean', 'std']).reset_index()

scatter_data['DATA_STR'] = scatter_data['DATA'].dt.strftime('%Y-%m')

# 2. Filtrar os meses
df_2022_06 = scatter_data[scatter_data['DATA_STR'] == '2022-06']
df_2026_03 = scatter_data[scatter_data['DATA_STR'] == '2026-03']

# 3. Criar subplots
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Junho/2022", "Março/2026"),
    vertical_spacing=0.12
)

# ---- GRÁFICO 1 (Jun/2022) ----
fig1 = px.scatter(
    df_2022_06,
    x='mean',
    y='std',
    color='REGIAO',
    hover_name='ESTADO',
    text='ESTADO',
    size='mean'
)

for trace in fig1.data:
    fig.add_trace(trace, row=1, col=1)

# Linhas de referência
fig.add_vline(x=df_2022_06['mean'].mean(), line_dash="dash", line_color="black", row=1, col=1)
fig.add_hline(y=df_2022_06['std'].mean(), line_dash="dash", line_color="black", row=1, col=1)

# ---- GRÁFICO 2 (Mar/2026) ----
fig2 = px.scatter(
    df_2026_03,
    x='mean',
    y='std',
    color='REGIAO',
    hover_name='ESTADO',
    text='ESTADO',
    size='mean'
)

for trace in fig2.data:
    fig.add_trace(trace, row=2, col=1)

# Linhas de referência
fig.add_vline(x=df_2026_03['mean'].mean(), line_dash="dash", line_color="black", row=2, col=1)
fig.add_hline(y=df_2026_03['std'].mean(), line_dash="dash", line_color="black", row=2, col=1)

# 4. Estética 
fig.update_traces(
    textposition='top center',
    cliponaxis=False,
    marker=dict(
        line=dict(width=1.5, color='white'),
        opacity=0.85
    ),
    textfont=dict(
        family="Arial Black",
        size=12
    )
)

fig.update_layout(
    width=1000,
    height=1000,
    template='plotly_white',
    title='Dinâmica de Mercado: Comparação Jun/2022 vs Mar/2026',
    title_font_size=24,
    legend=dict(title_font_size=16, font_size=14),
    xaxis=dict(
        showgrid=True,
        gridcolor='whitesmoke',
        linecolor='black',
        tickprefix="R$ "
    ),
    xaxis2=dict(
        showgrid=True,
        gridcolor='whitesmoke',
        linecolor='black',
        tickprefix="R$ "
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='whitesmoke',
        linecolor='black'
    ),
    yaxis2=dict(
        showgrid=True,
        gridcolor='whitesmoke',
        linecolor='black'
    )
)

fig.write_html('comparacao_dispersao.html')
fig.show()

In [ ]:

### Observações


O gráfico cruza **preço médio** (eixo X) com **coeficiente de variação** (eixo Y), 
revelando dois comportamentos distintos entre os estados.

**Junho/2022 (pico):**
- Alta dispersão generalizada — mercado sob choque, postos repassando de forma irregular
- **AP** destaca-se como outlier claro: preço baixo e dispersão baixa — política tributária protegeu o consumidor
- **BA** e **PA** combinam preço alto com alta dispersão — mercado fragmentado no pico
- **RR** e **TO** mostram preço alto mas dispersão baixíssima — mercados pequenos e homogêneos

**Março/2026:**
- Dispersão aumentou em quase todos os estados — mercado mais heterogêneo que em 2022
- **AM** e **AC** isolados no quadrante caro + disperso — Norte sofrendo mais com o choque recente
- **AP** continua barato, mas dispersão aumentou — efeito tributário ainda presente mas menos eficaz
- **DF** chama atenção: preço médio e dispersão baixíssima — mercado muito concentrado e competitivo

## 7. Gasolina e o Mercado Internacional: Correlação com o Brent

O preço da gasolina no Brasil não é determinado apenas por fatores internos — ele responde a choques do mercado internacional de petróleo, mediados pelo câmbio e pela política de preços da Petrobras.

Esta seção analisa quatro períodos estratégicos, dois de alta e dois de queda, para investigar como e com que velocidade o preço do Brent se transmite ao consumidor brasileiro.

### Períodos analisados

| Período | Evento | Direção |
|---------|--------|---------|
| Jan–Mar 2022 | Guerra na Ucrânia — choque de oferta global | 📈 Alta |
| Jun–Ago 2022 | Redução do ICMS — intervenção fiscal | 📉 Queda |
| Jan–Mar 2025 | Queda do Brent sem resposta da gasolina | 📉 Queda |
| Jan–Mar 2026 | Tensão no Oriente Médio — novo choque geopolítico | 📈 Alta |

A hipótese central testada é o efeito **_rockets and feathers_**: o repasse de alta é rápido e claro, enquanto o repasse de queda é lento ou inexistente.

In [ ]:
import yfinance as yf

# Baixando Brent (Petróleo) e Dólar (USDBRL)
tickers = ["BZ=F", "USDBRL=X"]
external_data = yf.download(tickers, start="2022-01-01", end="2026-04-24")['Close']

# Criando a coluna Brent em Reais (o que realmente impacta a Petrobras)
external_data['BRENT_BRL'] = external_data['BZ=F'] * external_data['USDBRL=X']

# Resampling para fechar com a sua frequência mensal (ME)
external_monthly = external_data['BRENT_BRL'].resample('ME').mean().reset_index()

In [ ]:
import yfinance as yf

# Baixando Brent e Dólar
tickers = ["BZ=F", "USDBRL=X"]
external_data = yf.download(tickers, start="2022-01-01", end="2026-04-24")['Close']
external_data['BRENT_BRL'] = external_data['BZ=F'] * external_data['USDBRL=X']

# Mensal — pra gráfico do período completo
external_monthly = external_data['BRENT_BRL'].resample('ME').mean()

# Diário — pra gráficos dos períodos de destaque
brent_diario = external_data[['BRENT_BRL']].copy()
brent_diario.index = pd.to_datetime(brent_diario.index)

preco_mensal_idx = preco_mensal.set_index('DATA')['PRECO']
datas_diarias = pd.date_range(start='2022-01-01', end='2026-04-24', freq='D')
gasolina_diario = preco_mensal_idx.reindex(datas_diarias).interpolate(method='time')

df_diario = pd.DataFrame({
    'GASOLINA': gasolina_diario,
    'BRENT_BRL': brent_diario['BRENT_BRL']
}).dropna()

print(df_diario.shape)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Criar a estrutura com eixo secundário
fig_duplo = make_subplots(specs=[[{"secondary_y": True}]])

# 2. Adicionar a Gasolina (Eixo Y Principal - Esquerda)
fig_duplo.add_trace(
    go.Scatter(
        x=preco_mensal['DATA'], 
        y=preco_mensal['PRECO'], 
        name="Gasolina (R$/L)",
        line=dict(color='firebrick', width=3)
    ),
    secondary_y=False,
)

# 3. Adicionar o Brent em R$ (Eixo Y Secundário - Direita)
fig_duplo.add_trace(
    go.Scatter(
        x=external_monthly['Date'], # Note que o yfinance costuma nomear como 'Date'
        y=external_monthly['BRENT_BRL'], 
        name="Brent (R$/Barril)",
        line=dict(color='royalblue', width=2, dash='dot')
    ),
    secondary_y=True,
)

# 4. Ajustes de Layout e Títulos
fig_duplo.update_layout(
    title='<b>Análise de Correlação: Combustível vs Mercado Internacional</b>',
    xaxis_title='Meses',
    template='plotly_white',
    width=1000,
    height=500,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Nomear os eixos para não confundir R$ 6,00 com R$ 500,00
fig_duplo.update_yaxes(title_text="<b>Gasolina</b> (R$/L)", secondary_y=False)
fig_duplo.update_yaxes(title_text="<b>Brent</b> (R$/Barril)", secondary_y=True)

fig_duplo.show()

In [ ]:
# Gasolina diária real
gasolina_diaria_real = gasolina.groupby('DATA')['PRECO'].mean()

df_diario_real = pd.DataFrame({
    'GASOLINA': gasolina_diaria_real,
    'BRENT_BRL': brent_diario['BRENT_BRL']
}).dropna()

# Períodos
periodos = [
    ('2022-01-01', '2022-03-31', 'Jan–mar 2022 — Subida (Guerra na Ucrânia)'),
    ('2022-06-01', '2022-08-31', 'Jun–Ago 2022 — Queda (Redução do ICMS)'),
    ('2025-01-01', '2025-03-31', 'Jan–Mar 2025 — Queda do Brent sem resposta'),
    ('2026-01-01', '2026-04-17', 'Jan–Mar 2026 — Subida (Tensão no Oriente Médio)'),
]

for inicio, fim, titulo in periodos:
    df_p = df_diario_real[(df_diario_real.index >= inicio) & 
                          (df_diario_real.index <= fim)].copy()
    df_p['GASOLINA_MM3'] = df_p['GASOLINA'].rolling(3, center=True).mean()
    
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    
    fig.add_trace(
        go.Scatter(x=df_p.index, y=df_p['GASOLINA_MM3'],
                   name="Gasolina (R$/L) — MM3",
                   line=dict(color='firebrick', width=3)),
        secondary_y=False)
    
    fig.add_trace(
        go.Scatter(x=df_p.index, y=df_p['BRENT_BRL'],
                   name="Brent (R$/Barril)",
                   line=dict(color='royalblue', width=2, dash='dot')),
        secondary_y=True)
    
    fig.update_layout(
        title=f'<b>Gasolina vs Brent — {titulo}</b>',
        xaxis_title='Data', template='plotly_white',
        width=1000, height=500,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))
    
    fig.update_yaxes(title_text="<b>Gasolina</b> (R$/L)", secondary_y=False)
    fig.update_yaxes(title_text="<b>Brent</b> (R$/Barril)", secondary_y=True)
    fig.write_html(f'brent_{inicio[:7]}.html')
    fig.show()

### 📈 Período 1: Jan–Mar 2022 — Guerra na Ucrânia

Em 24 de fevereiro de 2022, a Rússia invadiu a Ucrânia, desencadeando um choque de oferta no mercado global de petróleo. O Brent disparou de ~USD 85 para ~USD 130/barril em menos de três semanas — uma das altas mais abruptas da história recente.

O gráfico acima mostra como o preço da gasolina no Brasil respondeu a esse choque, com uma defasagem visível entre o pico do Brent e o repasse ao consumidor.

A análise de correlação com lag, calculada abaixo, estima um **repasse médio de 8 dias** com correlação de 0.85 — indicando que, neste período, a cada R$1 de alta no Brent, o consumidor brasileiro sentia o impacto no posto cerca de uma semana depois. 


In [ ]:
brent_jan_mar_2022 = df_diario_real[(df_diario_real.index >= '2022-01-01') & 
                                     (df_diario_real.index <= '2022-03-31')]['BRENT_BRL']
gasolina_completa = df_diario_real['GASOLINA'].rolling(3, center=True).mean()

lags = range(0, 61)
correlacoes = [brent_jan_mar_2022.corr(gasolina_completa.shift(-lag).reindex(brent_jan_mar_2022.index)) 
               for lag in lags]

melhor_lag = correlacoes.index(max(correlacoes))

plt.figure(figsize=(12, 5))
plt.plot(lags, correlacoes)
plt.axvline(x=melhor_lag, color='red', linestyle='--', label=f'Lag: {melhor_lag} dias')
plt.title('Correlação com lag — Brent vs Gasolina (Jan-Mar 2022)')
plt.xlabel('Lag (dias)')
plt.ylabel('Correlação')
plt.legend()
plt.grid(True)
plt.savefig('lag_2022.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Lag: {melhor_lag} dias | Correlação: {max(correlacoes):.3f}')

### 📉 Período 2: Jun–Ago 2022 — Redução do ICMS

Em junho de 2022, o governo federal reduziu o ICMS sobre combustíveis, derrubando o preço da gasolina em semanas. Este é um caso atípico — a queda não foi causada pelo mercado internacional, mas por intervenção fiscal direta.

O Brent permanecia elevado enquanto a gasolina despencava, invertendo temporariamente a relação histórica entre as duas séries. Por isso, a análise de lag não se aplica a este período.

### 📉 Período 3: Jan–Mar 2025 — Rockets and Feathers

Entre janeiro e março de 2025, o Brent caiu aproximadamente 20%, de R$500 para R$400/barril. O esperado seria uma queda proporcional no preço da gasolina. O que os dados mostram é o oposto: a gasolina subiu no mesmo período.

Este padrão tem nome na literatura econômica: **rockets and feathers**. A metáfora descreve a assimetria no repasse de preços de combustível — quando o petróleo sobe, o preço na bomba acompanha rápido, como um foguete. Quando o petróleo cai, o repasse é lento, parcial, ou simplesmente não acontece, como uma pena caindo.

No período analisado, a correlação entre Brent e gasolina é praticamente inexistente,confirmando que a queda do petróleo não se traduziu em alívio para o consumidor brasileiro. 

### 📈 Período 4: Jan–Mar 2026 — Tensão no Oriente Médio

No início de 2026, tensões geopolíticas no Oriente Médio pressionaram novamente o mercado de petróleo. O Brent saiu de ~R$340 para ~R$600/barril entre fevereiro e março, e a gasolina respondeu rapidamente.

Comparando com 2022, o repasse foi mais veloz — estimativa de lag de 4 dias contra 8 dias na guerra da Ucrânia — sugerindo que o mercado brasileiro incorpora choques externos com maior agilidade do que há quatro anos.

In [ ]:
brent_jan_mar_2026 = df_diario_real[(df_diario_real.index >= '2026-01-01') & 
                                     (df_diario_real.index <= '2026-03-31')]['BRENT_BRL']

correlacoes = [brent_jan_mar_2026.corr(gasolina_completa.shift(-lag).reindex(brent_jan_mar_2026.index)) 
               for lag in lags]

melhor_lag = correlacoes.index(max(correlacoes))

plt.figure(figsize=(12, 5))
plt.plot(lags, correlacoes)
plt.axvline(x=melhor_lag, color='red', linestyle='--', label=f'Lag: {melhor_lag} dias')
plt.title('Correlação com lag — Brent vs Gasolina (Jan-Mar 2026)')
plt.xlabel('Lag (dias)')
plt.ylabel('Correlação')
plt.legend()
plt.grid(True)
plt.savefig('lag_2026.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Lag: {melhor_lag} dias | Correlação: {max(correlacoes):.3f}')

### 🗺️ Período 4 — Análise por Estado: Velocidade de Repasse

A análise nacional estima um lag médio de 4 dias para o choque de 2026. Mas esse repasse é uniforme em todo o Brasil?

Estados com melhor infraestrutura logística, maior concentração de distribuidoras ou políticas tributárias específicas podem absorver e repassar choques de forma diferente. A análise abaixo calcula o lag de repasse individualmente para cada estado, revelando quais mercados respondem mais rápido e quais demoram mais.

In [ ]:

from scipy.signal import find_peaks

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

resultados_estado = []
estados = gasolina['ESTADO'].unique()

for estado in estados:
    # Filtro e Agrupamento
    gasolina_estado = gasolina[
        (gasolina['ESTADO'] == estado) & 
        (gasolina['DATA'] >= '2026-01-01') & 
        (gasolina['DATA'] <= '2026-03-31')
    ].groupby('DATA')['PRECO'].mean()
    
    if len(gasolina_estado) < 20:
        continue
    
    # Interpolação e Média Móvel
    datas_diarias = pd.date_range('2026-01-01', '2026-03-31', freq='D')
    gasolina_diario_estado = gasolina_estado.reindex(datas_diarias).interpolate(method='time')
    gasolina_mm3 = gasolina_diario_estado.rolling(3, center=True).mean()
    
    # Correlação vs Brent
    brent = df_jan_mar_2026['BRENT_BRL']
    lags = range(0, 21)
    correlacoes = np.array([brent.corr(gasolina_mm3.shift(-lag).reindex(brent.index)) for lag in lags])
    
    # Lógica do Primeiro Pico
    picos, _ = find_peaks(correlacoes)
    
    if len(picos) > 0:
        melhor_lag = picos[0]
        corr_pico = correlacoes[melhor_lag]
    else:
        melhor_lag = np.argmax(correlacoes)
        corr_pico = correlacoes[melhor_lag]

    # Regra da Correlação Mínima (0.7)
    lag_final = melhor_lag if corr_pico >= 0.7 else "Sem Correlação"

    resultados_estado.append({
        'ESTADO': estado,
        'LAG_DIAS': lag_final,
        'CORRELACAO': round(corr_pico, 4)
    })

# Criação do DataFrame, ordenação e Reset do Índice
df_resultado = pd.DataFrame(resultados_estado).sort_values('ESTADO').reset_index(drop=True)

# Exibição sem truncar
display(df_resultado.head())
display(df_resultado)

In [ ]:
fig, axes = plt.subplots(6, 5, figsize=(18, 20))
axes = axes.flatten()

for i, estado in enumerate(estados):
    gasolina_estado = gasolina[
        (gasolina['ESTADO'] == estado) & 
        (gasolina['DATA'] >= '2026-01-01') &
        (gasolina['DATA'] <= '2026-03-31')
    ].groupby('DATA')['PRECO'].mean()
    
    if len(gasolina_estado) < 20:
        axes[i].set_visible(False)
        continue
    
    datas_diarias = pd.date_range('2026-01-01', '2026-03-31', freq='D')
    gasolina_diario = gasolina_estado.reindex(datas_diarias).interpolate(method='time')
    gasolina_mm3 = gasolina_diario.rolling(3, center=True).mean()
    
    brent = df_diario_real[(df_diario_real.index >= '2026-01-01') & 
                           (df_diario_real.index <= '2026-03-31')]['BRENT_BRL']
    
    lags = range(0, 21)
    correlacoes = [brent.corr(gasolina_mm3.shift(-lag).reindex(brent.index)) for lag in lags]
    correlacoes_arr = np.array(correlacoes)
    
    picos = argrelmax(correlacoes_arr, order=2)[0]
    melhor_lag = int(picos[0]) if len(picos) > 0 else int(np.argmax(correlacoes_arr))
    
    ax = axes[i]
    ax.plot(lags, correlacoes)
    ax.axvline(melhor_lag, linestyle='--', color='red')
    ax.set_title(f'{estado} (lag={melhor_lag})', fontsize=10)
    ax.set_ylim(0, 1)

plt.suptitle('Lag de repasse do Brent por estado — Jan-Mar 2026', fontsize=14)
plt.tight_layout()
plt.savefig('lag_grid_estados.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df_plot = df_resultado_mapa.copy()
df_plot['LAG_LABEL'] = df_plot['LAG_DIAS'].astype(str)
df_plot = df_plot.sort_values('LAG_PLOT')

fig = px.bar(df_plot, x='ESTADO', y='LAG_PLOT', color='REGIAO',
             title='Lag de repasse do Brent por estado — Jan-Mar 2026',
             labels={'LAG_PLOT': 'Lag (dias)', 'ESTADO': 'Estado', 'REGIAO': 'Região'},
             template='plotly_white',
             hover_data=['CORRELACAO'])

fig.add_hline(y=df_plot['LAG_PLOT'].mean(), line_dash='dash', line_color='black',
              annotation_text=f'Média: {df_plot["LAG_PLOT"].mean():.1f} dias',
              annotation_position='top right')

fig.update_layout(width=0, height=500)
fig.write_html('lag_estados_bar.html')
fig.show()

### Observações Finais — Lag de Repasse por Estado

O gráfico revela que o repasse do choque do Brent não é uniforme no Brasil, com lag variando de **1 a 10 dias** entre os estados. A média nacional ficou em **4.6 dias** para o período de Jan-Mar 2026.

**Destaques:**

- **TO, PA, AM** — lag de 1 dia, os mercados mais rápidos. Surpreendente dado o isolamento logístico do Norte — pode indicar repasse especulativo imediato, antes mesmo da chegada física do combustível mais caro
- **MG e PR** — lag de 10 dias, os mais lentos. MG em particular apresenta correlação alta desde o início da curva, sugerindo que o mercado já precifica o choque gradualmente antes do pico formal de correlação
- **DF, GO, SC** — correlação abaixo de 0.70, repasse não identificado — dinâmica própria, possivelmente influência tributária ou amostragem insuficiente no período
- **Nordeste** — comportamento homogêneo, maioria entre 3 e 6 dias, correlação consistentemente alta

**Limitação importante:** a análise usa dados diários interpolados do mensal para a gasolina, o que suaviza variações intradiárias. Os lags estimados devem ser interpretados como aproximações, não valores exatos.

## Conclusão

Esta análise explorou o comportamento do preço da gasolina no Brasil entre 2022 e 2026, combinando dados da ANP com indicadores do mercado internacional.

**Principais achados:**

- O preço da gasolina no Brasil é fortemente influenciado pelo Brent, com lag médio estimado de **8 dias em 2022** e **4 dias em 2026** — sugerindo que o repasse ficou mais rápido ao longo dos anos
- O efeito **rockets and feathers** foi confirmado empiricamente: o repasse de alta é rápido e correlacionado, enquanto quedas do Brent não se traduzem em alívio proporcional ao consumidor
- Existe forte heterogeneidade regional — **Acre e Roraima** sistematicamente mais caros, **Amapá** como anomalia positiva de política tributária, **Sul e Sudeste** com mercado mais homogêneo e estável
- O lag de repasse varia de **1 a 10 dias entre estados**, sem padrão geográfico claro — sugerindo que fatores tributários e de mercado local importam tanto quanto logística

**Possíveis melhorias futuras:**

- Incorporar dados de ICMS por estado para isolar efeito tributário do efeito logístico
- Aplicar teste de Granger para formalizar a relação causal entre Brent e gasolina
- Expandir análise para etanol e diesel, investigando se o padrão rockets and feathers se repete
- Construir modelo preditivo de preço com base no Brent defasado, câmbio e sazonalidade